# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets by their @id
print("Available record sets (@id):")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name', '')}")

# Show all fields and columns for each record set
for rs in record_sets:
    print(f"\nRecord set @id: {rs['@id']}")
    fields = rs['fields'] if 'fields' in rs else []
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    @id: {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
    columns = rs['columns'] if 'columns' in rs else []
    if columns:
        print("  Columns:")
        for col in columns:
            print(f"    @id: {col['@id']} (name: {col.get('name', '')}, dataType: {col.get('dataType', '')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract record set @ids for demonstration
record_set_ids = [rs["@id"] for rs in dataset.metadata.record_sets]
print(f"Record set ids: {record_set_ids}")

# We will use the first record set for this analysis
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
print(f"Using main record set: {main_record_set_id}")

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record set {rs_id} ...")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading records for {rs_id}: {e}")

# Show the columns of the main DataFrame
if main_record_set_id in dataframes:
    print(f"Columns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Helper function to infer numeric fields
def find_numeric_fields(df):
    return [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if main_record_set_id in dataframes:
    df_main = dataframes[main_record_set_id]
    # Try to automatically select a numeric field for demonstration
    numeric_fields = find_numeric_fields(df_main)
    print(f"Numeric fields: {numeric_fields}")
    numeric_field = numeric_fields[0] if numeric_fields else None

    if numeric_field:
        threshold = df_main[numeric_field].quantile(0.75)  # e.g., above the 75th percentile
        print(f"Filtering records with {numeric_field} > {threshold}")
        filtered_df = df_main[df_main[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping
        # Automatically pick a non-numeric column
        group_field = next((col for col in df_main.columns if pd.api.types.is_object_dtype(df_main[col]) and col != numeric_field), None)

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id in dataframes and numeric_field:
    df_main = dataframes[main_record_set_id]

    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot if possible
    numeric_fields_2 = find_numeric_fields(df_main)
    if len(numeric_fields_2) >= 2:
        plt.figure(figsize=(6, 6))
        x_field, y_field = numeric_fields_2[0], numeric_fields_2[1]
        sns.scatterplot(x=df_main[x_field], y=df_main[y_field])
        plt.xlabel(x_field)
        plt.ylabel(y_field)
        plt.title(f"Scatter plot of {x_field} vs {y_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* This notebook demonstrated loading a Croissant-annotated dataset using the `mlcroissant` library.
* Dataset metadata, structure, and available fields were explored by referencing their `@id` as per Croissant schema best practices.
* Basic exploratory data analysis and visualizations were performed after loading the main record set.

Continue with additional domain-specific analysis as needed based on your research or application focus.